# ДЗ №3. Задача классификации с использованием полносвязной НС

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings('ignore')

In [4]:
import re

TEMP_MAP = {"O": 0, "B": 1, "A": 2, "F": 3, "G": 4, "K": 5, "M": 6}
LUM_MAP = {"I": 1, "II": 2, "III": 3, "IV": 4, "V": 5}

def parse_sptype(s):
    """
    На вход строка вида 'G5/G6V', 'M1V:', 'F3V', 'G3IV', 'G6III' и т.п.
    Возвращает (temp_code, subclass, lum_code_num)
    """
    if pd.isna(s):
        return np.nan, np.nan, np.nan
    s = str(s).strip().replace(" ", "").replace(":", "")
    temp_char = s[0]
    temp_code = TEMP_MAP.get(temp_char, np.nan)
    nums = re.findall(r"\d+", s)
    if len(nums) == 0:
        subclass = np.nan
    else:
        nums_float = [float(n) for n in nums]
        subclass = float(np.mean(nums_float))
    lum_code_num = np.nan
    for lum_str in ["III", "II", "IV", "V", "I"]:
        if lum_str in s:
            lum_code_num = LUM_MAP[lum_str]
            break

    return temp_code, subclass, lum_code_num

def bv_to_temp(bv):
    if pd.isna(bv):
        return np.nan
    return 4600 * (1/(0.92*bv + 1.7) + 1/(0.92*bv + 0.62))

def plx_to_distance(plx):
    if pd.isna(plx) or plx <= 0:
        return np.nan
    return 1000.0 / plx

def compute_absolute_mag(Vmag, dist_pc):
    if pd.isna(dist_pc) or dist_pc <= 0:
        return np.nan
    return Vmag - 5 * np.log10(dist_pc) + 5

In [5]:
train = pd.read_csv("train_star.csv")
test = pd.read_csv("test_star.csv")
train.head()

,Vmag,Plx,e_Plx,B-V,SpType,Amag,TargetClass
0,9.99,7.92,1.61,0.646,G5/G6V,19.483625,Giant
1,10.86,3.26,2.12,1.840,M1V:,18.426088,Giant
2,8.83,7.57,1.05,0.461,F3V,18.225480,Giant
3,7.72,24.80,0.89,0.613,G3IV,19.692257,Giant
4,8.81,3.17,1.03,0.872,G6III,16.315296,Dwarf


Заменим метки классов на числа: гигант - 1, карлик - 0

In [6]:
train = train.map(lambda value: {"Giant": 1, "Dwarf": 0}.get(value, value))
test = test.map(lambda value: {"Giant": 1, "Dwarf": 0}.get(value, value))

In [7]:
train[["Sp_temp", "Sp_subclass", "Sp_lum"]] = \
    train["SpType"].apply(lambda x: pd.Series(parse_sptype(x)))

test[["Sp_temp", "Sp_subclass", "Sp_lum"]] = \
    test["SpType"].apply(lambda x: pd.Series(parse_sptype(x)))

# Температура
train["T_eff"] = train["B-V"].apply(bv_to_temp)
test["T_eff"]  = test["B-V"].apply(bv_to_temp)

# Расстояние
train["Dist_pc"] = train["Plx"].apply(plx_to_distance)
test["Dist_pc"]  = test["Plx"].apply(plx_to_distance)

# Абсолютная величина M_V
train["M_V"] = train.apply(lambda r: compute_absolute_mag(r.Vmag, r.Dist_pc), axis=1)
test["M_V"]  = test.apply(lambda r: compute_absolute_mag(r.Vmag, r.Dist_pc), axis=1)

# Логарифмы
for col in ["T_eff", "Dist_pc"]:
    train[f"log_{col}"] = np.log10(train[col].clip(lower=1e-6))
    test[f"log_{col}"]  = np.log10(test[col].clip(lower=1e-6))


train = train.drop("SpType", axis=1)
test = test.drop("SpType", axis=1)

train.head()

,Vmag,Plx,e_Plx,B-V,Amag,TargetClass,Sp_temp,Sp_subclass,Sp_lum,T_eff,Dist_pc,M_V,log_T_eff,log_Dist_pc
0,9.99,7.92,1.61,0.646,19.483625,1,4.0,5.5,5.0,5793.079693,126.262626,4.483626,3.762910,2.101275
1,10.86,3.26,2.12,1.840,18.426088,1,6.0,1.0,5.0,3344.743474,306.748466,3.426088,3.524363,2.486782
2,8.83,7.57,1.05,0.461,18.225480,1,3.0,3.0,5.0,6571.226571,132.100396,3.225479,3.817646,2.120904
3,7.72,24.80,0.89,0.613,19.692257,1,4.0,3.0,4.0,5917.104412,40.322581,4.692258,3.772109,1.605548
4,8.81,3.17,1.03,0.872,16.315296,0,4.0,6.0,3.0,5072.687407,315.457413,1.315296,3.705238,2.498941


In [8]:
for col in ["Sp_temp", "Sp_subclass", "Sp_lum"]:
    train[col].fillna(train[col].median(), inplace=True)
    test[col].fillna(train[col].median(), inplace=True)
for col in ["Dist_pc", "M_V", "log_Dist_pc"]:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    test[col] = test[col].fillna(median_val)

### Создаём НС

In [9]:
X = train.drop(columns="TargetClass")
y = train["TargetClass"]

In [10]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

In [11]:
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

In [12]:
# Преобразование в тензоры PyTorch
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train.values)
X_val_tensor = torch.FloatTensor(X_val)
y_val_tensor = torch.LongTensor(y_val.values)

In [13]:
# Создание DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [14]:
# Определение модели полносвязной нейронной сети
class StarClassifier(nn.Module):
    def __init__(self, input_size, hidden_sizes=[128, 64, 32], dropout=0.3):
        super(StarClassifier, self).__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, 2))  # 2 класса: Dwarf и Giant
        layers.append(nn.CrossEntropyLoss())
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

In [15]:
# Функция для обучения модели
def train_model(model, train_loader, val_loader, epochs=100, lr=0.001, device='cpu'):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
    
    train_losses = []
    val_losses = []
    val_accuracies = []
    
    for epoch in range(epochs):
        # Обучение
        model.train()
        total_train_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.item()
        
        avg_train_loss = total_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Валидация
        model.eval()
        total_val_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                total_val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        
        avg_val_loss = total_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        val_accuracy = 100 * correct / total
        val_accuracies.append(val_accuracy)
        
        scheduler.step(avg_val_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.4f}, '
                  f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
    
    return train_losses, val_losses, val_accuracies

In [16]:
# Функция для предсказания
def predict(model, X_test_scaled, device='cpu'):
    model.eval()
    X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)
    
    with torch.no_grad():
        outputs = model(X_test_tensor)
        _, predicted = torch.max(outputs.data, 1)
    
    return predicted.cpu().numpy()

In [17]:
# Определение устройства
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [18]:
# Создание и обучение модели
input_size = X_train.shape[1]
model = StarClassifier(input_size, hidden_sizes=[256, 128, 64, 32], dropout=0.3)

In [19]:
# Обучение
train_losses, val_losses, val_accuracies = train_model(
    model, train_loader, val_loader, epochs=100, lr=0.001, device=device
)

TypeError: CrossEntropyLoss.forward() missing 1 required positional argument: 'target'

### Предсказание на тестовых данных

In [20]:
X_test = test.copy()
X_test_scaled = scaler.transform(X_test)

test_pred = predict(model, X_test_scaled, device=device)

submission = pd.DataFrame({
    "index": range(len(test_pred)),
    "TargetClass": test_pred
})

submission.to_csv("submission_6.csv", index=False)

TypeError: CrossEntropyLoss.forward() missing 1 required positional argument: 'target'